# 在 Colab 中打开
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH10/ch10_memory_topologies.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# 关于这个 Notebook

这个 Notebook 演示了：当父图（parent graph）在 LangGraph 中调用子 Agent 时，**检查点（checkpointing）机制会如何影响记忆行为**。

示例构建了一个简单的计费 worker，它会从用户的多轮输入中提取信息，例如是否发生重复扣费、用户的套餐等级，以及是否有退款意图。随后，一个父图会调用这个 worker，但每次只向它传递当前这一轮用户消息。Notebook 将三种执行模式并排比较：每次调用都创建全新 worker、按 thread 持久化的 worker，以及完全不使用 checkpoint 的无状态 worker。

核心目标是说明：**子 Agent 是否拥有记忆，取决于 worker graph 是如何编译和配置 checkpoint 的。** 即使父图每次只传递最新一轮消息，只要 worker 使用 checkpointer，它仍然可以在同一个 thread 内保留此前的消息。这使得这个 Notebook 很适合理解 LangGraph 多 Agent 系统中的记忆边界（memory boundary）、线程级持久化（thread-level persistence）以及子 Agent 状态管理。


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages


# 定义 worker 的状态与节点
class WorkerState(TypedDict):
    messages: Annotated[list, add_messages]


def billing_worker_node(state: WorkerState):
    human_texts = [
        m.content.lower()
        for m in state["messages"]
        if isinstance(m, HumanMessage)
    ]

    issue = "duplicate" if any("charged twice" in t for t in human_texts) else None
    tier = "pro" if any("pro plan" in t for t in human_texts) else None
    action = "refund" if any("refund" in t for t in human_texts) else None

    msg_count = len(state["messages"])
    answer = f"Msgs: {msg_count} | Known: {issue}, {tier}, {action}"
    return {"messages": [AIMessage(content=answer)]}


# 构建 worker graph
def build_worker(mode: str):
    builder = StateGraph(WorkerState)
    builder.add_node("billing_worker", billing_worker_node)
    builder.add_edge(START, "billing_worker")

    if mode == "per_invocation":
        return builder.compile()
    if mode == "per_thread":
        return builder.compile(checkpointer=True)
    if mode == "stateless":
        return builder.compile(checkpointer=False)

    raise ValueError(f"Unknown mode: {mode}")


# 父图每次只传递当前这一轮输入
class ParentState(TypedDict):
    current_input: str
    answer: str


def make_parent_app(worker_app):
    def call_worker(state: ParentState, config: RunnableConfig):
        out = worker_app.invoke(
            {"messages": [HumanMessage(content=state["current_input"])]},
            config=config,
        )
        return {"answer": out["messages"][-1].content}

    builder = StateGraph(ParentState)
    builder.add_node("call_worker", call_worker)
    builder.add_edge(START, "call_worker")
    return builder.compile(checkpointer=InMemorySaver())


# 将三种模式并排比较
def run_comparison():
    apps = {
        "Per-invocation (fresh start)": make_parent_app(build_worker("per_invocation")),
        "Per-thread (persistent)": make_parent_app(build_worker("per_thread")),
        "Stateless (no checkpoint)": make_parent_app(build_worker("stateless")),
    }

    turns = [
        "I was charged twice.",
        "I'm on the Pro plan.",
        "I want a refund.",
    ]

    results = []

    for label, app in apps.items():
        cfg = {"configurable": {"thread_id": f"demo-{label}"}}
        for i, turn in enumerate(turns, start=1):
            out = app.invoke({"current_input": turn}, cfg)
            results.append(
                {
                    "Mode": label,
                    "Turn": i,
                    "Input": turn,
                    "Subagent Output": out["answer"],
                }
            )

    print(f"{'MODE':<32} | {'TURN':<5} | {'SUBAGENT OUTPUT'}")
    print("-" * 95)
    for r in results:
        print(f"{r['Mode']:<32} | {r['Turn']:<5} | {r['Subagent Output']}")


run_comparison()

MODE                             | TURN  | SUBAGENT OUTPUT
-----------------------------------------------------------------------------------------------
Per-invocation (fresh start)     | 1     | Msgs: 1 | Known: duplicate, None, None
Per-invocation (fresh start)     | 2     | Msgs: 1 | Known: None, pro, None
Per-invocation (fresh start)     | 3     | Msgs: 1 | Known: None, None, refund
Per-thread (persistent)          | 1     | Msgs: 1 | Known: duplicate, None, None
Per-thread (persistent)          | 2     | Msgs: 3 | Known: duplicate, pro, None
Per-thread (persistent)          | 3     | Msgs: 5 | Known: duplicate, pro, refund
Stateless (no checkpoint)        | 1     | Msgs: 1 | Known: duplicate, None, None
Stateless (no checkpoint)        | 2     | Msgs: 1 | Known: None, pro, None
Stateless (no checkpoint)        | 3     | Msgs: 1 | Known: None, None, refund
